# Retrain on Colab

Rebuilds the model with the leak-free pipeline:
- imputation and frequency maps are fitted on train only
- the latest 20% of train is held out **before** SMOTE as validation
- Optuna tunes on validation, and the threshold is picked on validation
- the test set is scored once, at the end

Before running, add a Colab secret named `KAGGLE_API_TOKEN` (key icon in the left bar) and accept the competition rules at kaggle.com/c/ieee-fraud-detection/rules. A standard CPU runtime is enough; High-RAM helps.

In [2]:
!git clone -q https://github.com/Shashank17singh/Agentic-Fraud-Sentinel.git
%cd Agentic-Fraud-Sentinel
!git log --oneline -1
!pip install -q -r requirements.txt kaggle

/content/Agentic-Fraud-Sentinel
d369f5e (HEAD -> main, origin/main, origin/HEAD) Created using Colab


In [3]:
import os

from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!kaggle competitions download -c ieee-fraud-detection -p data/raw -q
!unzip -oq data/raw/ieee-fraud-detection.zip train_transaction.csv train_identity.csv -d data/raw
!ls -la data/raw

total 814216
drwxr-xr-x 2 root root      4096 Nov 22 23:50 .
drwxr-xr-x 3 root root      4096 Nov 22 23:50 ..
-rw-r--r-- 1 root root 123856947 Dec 11  2019 ieee-fraud-detection.zip
-rw-r--r-- 1 root root  26529680 Dec 11  2019 train_identity.csv
-rw-r--r-- 1 root root 683351067 Dec 11  2019 train_transaction.csv


In [6]:
import sys

# Same layout as the local notebooks: outputs land in notebooks/data/, where the API looks.
%cd notebooks
sys.path.insert(0, "..")

from src.data.preprocessing import run_preprocessing_pipeline

X_train, X_val, X_test, Y_train, Y_val, Y_test = run_preprocessing_pipeline(
    raw_transaction_path="../data/raw/train_transaction.csv",
    raw_identity_path="../data/raw/train_identity.csv",
)

/content/Agentic-Fraud-Sentinel/notebooks
Loading data...
Dropping 6 high-missing transaction cols
Dropping 9 high-missing identity cols
Loading transaction file...
Transaction: 1257.5 MB
Loading identity file...
Identity: 123.3 MB
Joined shape: (590540, 419)

Engineering features...

Splitting...
Split point: 12,192,854 seconds
Train: 472,432 rows | Fraud rate: 3.514%
Test:  118,108 rows  | Fraud rate: 3.441%

Frequency encoding...

Imputing (medians from train only)...

Encoding categoricals...
Encoded 29 categorical columns

Separating features and target...
X_train: 472,432 rows | Y_train: 472,432 rows
X_test:  118,108 rows  | Y_test:  118,108 rows
X_val:   94,487 rows (held out before SMOTE)

Applying SMOTE...
X_train rows: 377,945 | Y_train rows: 377,945
Before SMOTE � fraud: 12,895 / 377,945 (3.412%)
After SMOTE  � fraud: 36,505 / 401,555 (9.091%)

Saving...
Saved to data/processed/


In [7]:
import joblib
import pandas as pd

from src.explainability.shap_explainer import (
    build_explainer,
    compute_shap_values,
    get_global_importance,
)

if "tuned_xgb" not in globals():
    tuned_xgb = joblib.load("data/models/xgboost_production.pkl")["model"]

sample = X_test.sample(5000, random_state=42)
shap_values = compute_shap_values(
    build_explainer(tuned_xgb, X_train.sample(1000, random_state=42)), sample
)
top = get_global_importance(shap_values, sample.columns, top_n=20)

raw_cols = set(pd.read_csv("../data/raw/train_transaction.csv", nrows=0).columns) | set(
    pd.read_csv("../data/raw/train_identity.csv", nrows=0).columns
)
top["engineered"] = ~top["feature"].isin(raw_cols)

print(f"Engineered in top 6: {top.head(6)['engineered'].sum()} of 6")
print(f"Engineered in top 20: {top['engineered'].sum()} of 20")
top

Building TreeExplainer...
explainer built.
Computing shap values for 5,000 rows
SHAP values shape: (5000, 445)
Engineered in top 6: 1 of 6
Engineered in top 20: 5 of 20


,feature,importance,engineered
25,C13,0.326592,False
40,M4,0.302498,False
362,V317,0.281413,False
0,TransactionAmt,0.230956,False
418,tx_hour_sin,0.199118,True
13,C1,0.197277,False
26,C14,0.184856,False
416,tx_hour,0.181382,True
436,card1_freq,0.162692,True
303,V258,0.148755,False


In [ ]:
from src.models.evaluate import evaluate_model, log_results, select_threshold
from src.models.train import train_xgboost_baseline

baseline = train_xgboost_baseline(X_train, Y_train)
baseline_results = evaluate_model(baseline, X_test, Y_test, model_name="xgb_baseline")
log_results(baseline_results)

In [ ]:
from src.models.train import tune_xgboost

tuned_xgb, study = tune_xgboost(X_train, Y_train, X_val, Y_val, n_trials=50)
print("Best validation PR-AUC:", study.best_value)
print("Best params:", study.best_params)

In [ ]:
# Threshold comes from validation only; test is scored once with it.
best_threshold = select_threshold(Y_val, tuned_xgb.predict_proba(X_val)[:, 1])
print(f"Threshold chosen on validation: {best_threshold:.4f}")

tuned_results = evaluate_model(tuned_xgb, X_test, Y_test, model_name="xgb_tuned")
log_results(tuned_results)
tuned_results_optimal = evaluate_model(
    tuned_xgb,
    X_test,
    Y_test,
    threshold=best_threshold,
    model_name="xgb_tuned_val_threshold",
)
log_results(tuned_results_optimal)

In [ ]:
import joblib

from src.explainability.shap_explainer import build_explainer, save_explainer

os.makedirs("data/models", exist_ok=True)
# Saved for the CPU-only API; GPU training must not leak into inference.
tuned_xgb.set_params(device="cpu")
joblib.dump(
    {"model": tuned_xgb, "threshold": best_threshold, "metrics": tuned_results_optimal},
    "data/models/xgboost_production.pkl",
)
save_explainer(build_explainer(tuned_xgb, X_train.sample(1000, random_state=42)))

In [ ]:
# Download the new artifacts, then copy them over notebooks/data/ in your local repo.
!zip -qr /content/retrained.zip data/models data/model_results
from google.colab import files

files.download("/content/retrained.zip")